In [1]:
import Pkg
Pkg.activate("..")
Pkg.instantiate()

  Activating project at `~/repos/WaveGreen2D`


In [2]:
using WaveGreen2D: create_wave

In [4]:
using StaticArrays: Size

In [3]:
WaveGreen2D.create_wave(depth=5.0, frequency=0.8)

LoadError: UndefVarError: `WaveGreen2D` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
Hint: WaveGreen2D is loaded but not imported in the active module Main.

In [6]:
wave = create_wave(depth=5.0, frequency=6)

Infinite-depth wave: ω = 6.0 rad/s, g = 9.81 m/s²

In [4]:
wave = create_wave(depth=5.0, frequency=0.8)

Finite-depth wave: h = 5.0 m, ω = 0.8 rad/s, g = 9.81 m/s²

In [7]:
typeof(wave)

WaveGreen2D.InfiniteDepthWave

In [9]:
abstract type AbstractWaterWave end


# Number of evanescent modes
const _nevamodes = 20


"""
    FiniteDepthWave

Parameters that define a wave at waters of finite depth.

# Fields
- `h::Float64`: water depth (m)
- `ω::Float64`: wave frequency (rad/s)
- `g::Float64`: acceleration of gravity (m/s²)
- `K::Float64`: infinite-depth wavenumber (1/m⁻¹)
- `k₀::Float64`: wavenumber (1/m⁻¹)
- `kₙ::NTuple{n, Float64}`: n evanescent wavenumbers (1/m⁻¹)
- `L₁::ChebyshevSeries{Float64, 2}`: Near field integral L₁
- `L₂::ChebyshevSeries{Float64, 2}`: Near field integral L₂
"""
struct FiniteDepthWave <: AbstractWaterWave
    h::Float64
    ω::Float64
    g::Float64
    K::Float64
    k₀::Float64
    kₙ::NTuple{_nevamodes, Float64}
    L₁::ChebyshevSeries{Float64, 2}
    L₂::ChebyshevSeries{Float64, 2}
end


"""
    InfiniteDepthWave

Parameters that define a wave at waters of infinite depth.

# Fields
- `h::Float64`: water depth (m)
- `ω::Float64`: wave frequency (rad/s)
- `g::Float64`: acceleration of gravity (m/s²)
- `K::Float64`: wavenumber (1/m⁻¹)
"""
struct InfiniteDepthWave <: AbstractWaterWave
    h::Float64
    ω::Float64
    g::Float64
    K::Float64
end


# Avoid non-physical values for the wave parameters.
function validate_wave(depth::Real, frequency::Real, gravity::Real)
    if depth ≤ 0
        throw(DomainError(depth, "The depth must be positive."))
    elseif frequency < 0
        throw(DomainError(frequency, "The frequency must be non-negative."))
    elseif gravity ≤ 0
        throw(DomainError(gravity, "The acceleration of gravity must be positive."))
    end
end


"""
    create_wave(; depth::Real, frequency::Real, gravity::Real=9.80665) -> AbstractWaterWave

Creates the wave by defining its frequency and the environmental conditions.
"""
function create_wave(; depth::Real, frequency::Real, gravity::Real=9.80665)
    h = Float64(depth)
    ω = Float64(frequency)
    g = Float64(gravity)

    validate_wave(h, ω, g)

    K = ω^2 / g
    H = K * h
    
    if 0.01 ≤ H ≤ π
        k₀ = find_k₀(h, ω, g)
        kₙ = ntuple(i -> find_kₙ(i, h, ω, g), _nevamodes)
        L₁, L₂ = NearField.reduce_integrals(H)
        return FiniteDepthWave(h, ω, g, K, k₀, kₙ, L₁, L₂)
    else
        return InfiniteDepthWave(h, ω, g, K)
    end
end


"""
    find_k₀(h::Real, ω::Real, g::Real=9.80665) -> Float64

Finds the wavenumber `k₀` from the dispersion relation ``ω^2 = k g \\tanh(k h)``.
"""
function find_k₀(h::Real, ω::Real, g::Real=9.80665)
    f(k::Real) = k*g * tanh(k*h) - ω^2
    f′(k::Real) = g * tanh(k*h) + k*g*h * sech(k*h)^2

    # Initial estimate for k₀
    κ₁ = ω/√(g*h)  # shallow water
    κ₂ = ω^2/g  # deep water
    k̄₀ = √(κ₁*κ₂)  # geometric mean

    return findroot(f, f′, k̄₀)
end


"""
    find_kₙ(n::Int, h::Real, ω::Real, g::Real=9.80665) -> Float64

Finds the evanescent wavenumber `kₙ` from the dispersion relation ``ω^2 = -k g \\tan(k h)``.
"""
function find_kₙ(n::Int, h::Real, ω::Real, g::Real=9.80665)
    f(k::Real) = k*g * tan(k*h) + ω^2
    f′(k::Real) = g * tan(k*h) + k*g*h * sec(k*h)^2

    # Initial estimate for kₙ
    κ₁ = (2n-1)*π/2h
    κ₂ = n*π/h
    k̄ₙ = 0.5*(κ₁+κ₂)

    return findroot(f, f′, k̄ₙ)
end


function Base.show(io::IO, ::MIME"text/plain", w::FiniteDepthWave)
    print(
        io,
        "Finite-depth wave ",
        "h = $(round(w.h; digits=2)) m, ",
        "ω = $(round(w.ω; digits=2)) rad/s, ",
        "g = $(round(w.g; digits=2)) m/s²",
    )
end


function Base.show(io::IO, ::MIME"text/plain", w::InfiniteDepthWave)
    print(
        io,
        "Infinite-depth wave ",
        "ω = $(round(w.ω; digits=2)) rad/s, ",
        "g = $(round(w.g; digits=2)) m/s²",
    )
end


function Base.show(io::IO, w::AbstractWaterWave)
    Base.show(io, MIME"text/plain"(), w)
end;

In [10]:
wave = create_wave(depth=5.0, frequency=0.8)

LoadError: UndefVarError: `reduce_integrals` not defined in `WaveGreen2D.NearField`
Suggestion: check for spelling errors or missing imports.

Wave parameters: h = 5.0 m, ω = 6.0 rad/s, k₀ = 3.67 m, g = 9.81 m/s²

In [24]:
wave

Wave parameters: h = 5.0 m, ω = 6.0 rad/s, k₀ = 3.67 m, g = 9.81 m/s²

In [18]:
WaveParameters(1.0, 2.0, 3.0, 4.0)

WaveParameters(1.0, 2.0, 3.0, 4.0)

In [19]:
setwave!(depth=5.0, wavenumber=6.0)

WaveParameters(5.0, 7.6707170460133645, 6.0, 9.80665)

In [23]:
setwave!(depth=5.0)

LoadError: Either frequency or wavenumber must be given.

In [24]:
setwave!(depth=5.0, frequency=6.0, wavenumber=2.0)

LoadError: Either frequency or wavenumber must be given.

In [25]:
@code_warntype setwave!(depth=5.0, frequency=6.0)

MethodInstance for Core.kwcall(::@NamedTuple{depth::Float64, frequency::Float64}, ::typeof(setwave!))
  from kwcall(::NamedTuple, ::typeof(setwave!)) @ Main In[14]:39
Arguments
  _::Core.Const(Core.kwcall)
  @_2::@NamedTuple{depth::Float64, frequency::Float64}
  @_3::Core.Const(Main.setwave!)
Locals
  @_4::Union{Nothing, Float64}
  depth::Float64
  frequency::Float64
  wavenumber::Nothing
  gravity::Float64
Body::WaveParameters
1 ──       Core.NewvarNode(:(@_4))
│    %2  = Core.isdefined(@_2, :depth)::Core.Const(true)
└───       goto #6 if not %2
2 ── %4  = Core.getfield(@_2, :depth)::Float64
│    %5  = Main.Real::Core.Const(Real)
│    %6  = (%4 isa %5)::Core.Const(true)
└───       goto #4 if not %6
3 ──       goto #5
4 ──       Core.Const(:(Main.Real))
│          Core.Const(:(%new(Core.TypeError, Symbol("keyword argument"), :depth, %9, %4)))
└───       Core.Const(:(Core.throw(%10)))
5 ┄─       (@_4 = %4)
└───       goto #7
6 ──       Core.Const(:(Core.UndefKeywordError(:depth)))
└─── 

In [26]:
wave.ω

7.6707170460133645

In [27]:
wave.h

5.0

In [28]:
wave.k₀

6.0

In [29]:
wave.g

9.80665